# DonkeySim Manual Data Collection

Drive the car yourself and save images for VAE training.

Prereqs:
- Start DonkeySim with a GUI (for example: `./docker/simulator/sim.sh sim --no-headless --sim-port 9091`).
- Run this notebook inside the trainer image (or a Python env with `gym_donkeycar`, `imageio`, `ipywidgets`).
- Use the sliders to steer/throttle; toggle the **record** button to start/stop saving frames.


In [ ]:
import asyncio
from pathlib import Path

import imageio
import gym, gym_donkeycar  # preinstalled in the trainer image
import ipywidgets as widgets
from IPython.display import display

# Connection and track selection
HOST = "localhost"
PORT = 9091
ENV_ID = "donkey-generated-track-v0"  # change to e.g. donkey-warehouse-v0, donkey-mountain-track-v0, ...

# Output location
OUT_DIR = Path("dataset_manual")
OUT_DIR.mkdir(exist_ok=True)

conf = {"exe_path": "remote", "host": HOST, "port": PORT, "cam_resolution": (160, 120)}
env = gym.make(ENV_ID, conf=conf)
obs = env.reset()
frame_id = len(list(OUT_DIR.glob("*.jpg")))

# Controls
steer = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.01, description="steer")
throttle = widgets.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.01, description="throttle")
record = widgets.ToggleButton(value=False, description="record", button_style="danger")
status = widgets.HTML("<b>Stopped</b>")
ui = widgets.VBox([widgets.HBox([steer, throttle, record]), status])
display(ui)

task = None

async def drive_loop():
    global obs, frame_id
    while record.value:
        obs, _, done, _ = env.step([float(steer.value), float(throttle.value)])
        imageio.imwrite(OUT_DIR / f"{frame_id:06d}.jpg", obs)
        frame_id += 1
        if done:
            obs = env.reset()
        await asyncio.sleep(0.05)  # ~20 FPS

def on_toggle(change):
    global task
    if change["new"]:
        status.value = "<b>Recording...</b>"
        task = asyncio.create_task(drive_loop())
    else:
        status.value = "<b>Stopped</b>"
        if task is not None:
            task.cancel()
            task = None

record.observe(on_toggle, names="value")


In [ ]:
# Optional: zip the collected dataset for Colab/transfer
import datetime, shutil

zip_name = f"{OUT_DIR.name}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(zip_name, "zip", OUT_DIR)
zip_name + ".zip"
